In [3]:
class MapColoringCSP:
    def __init__(self, variables, domains, neighbors):
        """
                :param variables: List of regions, e.g., ['WA', 'NT', 'SA', ...]
                :param domains: Dict mapping each region to a list of available colors
                :param neighbors: Dict mapping each region to a list of its neighboring regions
                """
        self.variables = variables
        self.domains = {v: list(domains[v]) for v in variables}
        self.neighbors = neighbors

    def is_consistent(self, var, color, assignment):
        """Check if assigning 'color' to 'var' violates constraints with neighbors."""
        for neighbor in self.neighbors[var]:
            if neighbor in assignment and assignment[neighbor] == color:
                return False
        return True

    def select_unassigned_variable(self, assignment):
        """
                MRV (Minimum Remaining Values) Heuristic:
                Select the unassigned variable with the fewest legal remaining colors.
                """
        unassigned = [v for v in self.variables if v not in assignment]
        return min(unassigned, key=lambda v: len(self.domains[v]))

    def backtrack(self, assignment):
        """Core CSP Backtracking Search with Forward Checking."""
        # Base case: All variables assigned
        if len(assignment) == len(self.variables):
            return assignment

        var = self.select_unassigned_variable(assignment)

        for color in list(self.domains[var]):
            if self.is_consistent(var, color, assignment):
                # Make assignment
                assignment[var] = color

                # Forward Checking: Save current domain states to allow pruning
                saved_domains = {v: list(self.domains[v]) for v in self.variables}
                pruned_successfully = True

                # Prune the selected color from neighbors' domains
                for neighbor in self.neighbors[var]:
                    if neighbor not in assignment:
                        if color in self.domains[neighbor]:
                            self.domains[neighbor].remove(color)
                            # Domain failure (empty domain created)
                            if not self.domains[neighbor]:
                                pruned_successfully = False
                                break

                if pruned_successfully:
                    result = self.backtrack(assignment)
                    if result is not None:
                        return result

                # Backtrack: Restore state
                del assignment[var]
                self.domains = saved_domains

        return None

    def solve(self):
        return self.backtrack({})


# ================= Example Usage: Map of Australia =================

# 1. Variables (States of Australia)
regions = ["WA", "NT", "SA", "Q", "NSW", "V", "T"]

# 2. Domains (Available colors for each region)
colors = ["Red", "Green", "Blue"]
domains = {region: colors.copy() for region in regions}

# 3. Constraints (Adjacency graph)
neighbors = {
    "WA":  ["NT", "SA"],
    "NT":  ["WA", "SA", "Q"],
    "SA":  ["WA", "NT", "Q", "NSW", "V"],
    "Q":   ["NT", "SA", "NSW"],
    "NSW": ["Q", "SA", "V"],
    "V":   ["SA", "NSW"],
    "T":   []  # Tasmania has no neighbors
}

csp = MapColoringCSP(regions, domains, neighbors)
solution = csp.solve()

if solution:
    print("Valid Map Coloring Found:\n")
    for region, color in solution.items():
        print(f"  {region:4s} -> {color}")
else:
    print("No valid coloring exists.")

Valid Map Coloring Found:

  WA   -> Red
  NT   -> Green
  SA   -> Blue
  Q    -> Red
  NSW  -> Green
  V    -> Red
  T    -> Red
